# Step 6 · How much does the cost ceiling matter?

So far the ceiling was "no more than the baseline". That rule was set *before* looking at results, which keeps it honest. But it is a business choice, so here we show what other ceilings would give:
- **0.75×:** a 25 % cheaper medium;
- **1.05×:** a 5 % higher budget. This includes the best recipe ever measured, which costs 1.6 % more than the baseline;
- **1.25×:** a 25 % higher budget.

For each ceiling we rerun the full pipeline from step 5.

**How to read the table** (one row per ceiling):
- **A. best measured y:** the highest cell count among the 25 *tested* recipes that fit within this ceiling.
- **B. bar to beat:** the model's estimate of that recipe's true growth (step 4). Used by Expected Improvement.
- **C-F** describe the **3 exploit picks** of the recommended batch, averaged:
  - C. predicted y (mean);
  - D. pessimistic y (lower end of the 90 % interval);
  - E. FBS level;
  - F. cost relative to the baseline.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd

from mediaopt.acquisition import recommend
from mediaopt.cost import baseline_cost, media_cost
from mediaopt.data import load_recipes

df = load_recipes()
BASE = baseline_cost()

rows = []
for factor in [0.75, 1.0, 1.05, 1.25]:
    ceiling = factor * BASE
    batch, bar = recommend(df, ceiling)
    exploit = batch[batch.role == "exploit"]
    rows.append({
        "ceiling (x baseline)": factor,
        "A. best measured y": df[media_cost(df) <= ceiling].y.max(),
        "B. bar to beat": bar,
        "C. predicted y": exploit["mean"].mean(),
        "D. pessimistic y": exploit.low90.mean(),
        "E. FBS (% v/v)": exploit.FBS.mean() * 17.5,
        "F. cost (x baseline)": exploit.cost.mean() / BASE,
    })
pd.DataFrame(rows).set_index("ceiling (x baseline)").round(2)

,A. best measured y,B. bar to beat,C. predicted y,D. pessimistic y,E. FBS (% v/v),F. cost (x baseline)
ceiling (x baseline),,,,,,
0.75,-1.62,-1.04,1.37,0.42,7.18,0.74
1.00,2.52,1.40,1.47,0.41,8.74,0.88
1.05,3.31,1.42,1.58,0.60,9.77,0.98
1.25,3.31,1.42,1.60,0.58,11.48,1.16


**Recommendation to the business**
- Keep **"no more than today's medium" (1.0×)** as the default. Its batch is already 12 % cheaper on average.
- Offer **+5 % (1.05×)** as an option. It includes the best recipe found so far and buys a small expected gain, while the batch still costs about the same as today on average.
- **Don't go beyond +5 %:** extra budget buys more serum, which no longer helps.